<a href="https://colab.research.google.com/github/VanesaCam/DATA-ANALYTICS-ML-CHALLENGE/blob/main/DATA-ANALYTICS-ML-CHALLENGE/notebooks%20/S12_ML_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA ANALYTICS ML CHALLENGE / 2026-2
## Laboratorio 1. Modelos supervisados y métricas (Semana 12)

Profesor: Elias Buitrago Bolivar  
Universidad Central  
Caso: Brazilian E-Commerce Public Dataset by Olist

Objetivo: construir un modelo base para anticipar entregas tardías a partir de datos de compras, productos, clientes, vendedores y pagos.

Las instrucciones de GitHub y entrega se encuentran en el documento técnico. Las secciones de carga y preparación se suministran resueltas; complete las cuatro actividades de modelado y evaluación.

## 1. Carga de datos

Una observación del modelo corresponde a un pedido, aunque tenga varios productos y pagos. Se integrarán siete tablas del conjunto público de Olist.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

In [2]:
ORIGEN = ('https://raw.githubusercontent.com/spdrio/'
          'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
          'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')

pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')

print('Pedidos:', pedidos.shape)
print('Items:', items.shape)
print('Productos:', productos.shape)
print('Clientes:', clientes.shape)
print('Vendedores:', vendedores.shape)
print('Pagos:', pagos.shape)

Pedidos: (99441, 8)
Items: (112650, 7)
Productos: (32951, 9)
Clientes: (99441, 5)
Vendedores: (3095, 4)
Pagos: (103886, 5)


## 2. Preparación de variables

Un pedido puede contener varios ítems y pagos. Primero se resumen estas tablas por `order_id`, para evitar contar un mismo pedido varias veces. Las variables de destino se construyen después y no se incluyen entre los predictores.

In [3]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [4]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

Pedidos integrados: 98666


In [5]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

Pedidos disponibles: 96470
Entregas tardías: 6.77 %


### Variables del modelo

Se utilizarán datos que razonablemente están disponibles al cierre de la compra: plazo prometido, calendario, contenido del pedido, dimensiones de los productos, ubicación y medio de pago. Las fechas reales de envío y entrega, y las reseñas posteriores, quedan excluidas de las variables predictoras.

La lista de variables y sus tablas de origen se encuentra en el diccionario técnico anexo.

In [6]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

Variables predictoras: 28
Numéricas: 24 | Categóricas: 4


### Muestra individual

Cada estudiante utilizará 35.000 pedidos, seleccionados a partir del ID numérico entregado por el profesor (por ejemplo, ID 01). El mismo ID genera siempre la misma muestra y las mismas particiones.

Se utilizarán 24.500 pedidos para entrenar, 5.250 para validar y 5.250 para la prueba final. Durante las semanas 12 a 15 no se utilizarán los datos de prueba para elegir modelos.

En la siguiente celda cambie `ID_ESTUDIANTE` por el número asignado por el profesor. Por ejemplo, el ID 01 se escribe como `1`.


In [7]:
# Escriba el ID que le asignó el profesor (01 se escribe 1).
ID_ESTUDIANTE = 5
assert isinstance(ID_ESTUDIANTE, int) and 1 <= ID_ESTUDIANTE <= 30, (
    'El ID debe ser un número entero entre 1 y 30.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)

Estudiante ID: 05 | Semilla: 1005
Muestra: 35000 | Train: 24500 | Valid: 5250 | Test: 5250
Huella de muestra: 6b93315f88e1


## 3. Modelo de referencia y clasificación

Las variables numéricas se imputan y escalan; las categóricas se transforman con *One-Hot Encoding*. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, dentro del `Pipeline`.

In [8]:
preprocesamiento = ColumnTransformer([
    ('numericas', make_pipeline(SimpleImputer(strategy='median'),
                               StandardScaler()), columnas_numericas),
    ('categoricas', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')),
     columnas_categoricas)
])

referencia = DummyClassifier(strategy='most_frequent')
referencia.fit(X_train, y_train)
pred_ref = referencia.predict(X_valid)

print('Accuracy referencia:', round(accuracy_score(y_valid, pred_ref), 3))
print('F1 referencia:', round(f1_score(y_valid, pred_ref, zero_division=0), 3))

Accuracy referencia: 0.932
F1 referencia: 0.0


### Actividad 1. Entrenamiento

Complete el entrenamiento de la regresión logística usando `preprocesamiento`. Pruebe `class_weight=None` o `'balanced'`, y explique su elección en la sección de interpretación. Conserve las variables `modelo` y `pred`.

En el estimador utilice `class_weight=peso_clases` y `max_iter=1000` como límite de iteraciones. Si aparece una advertencia de convergencia, revise el entrenamiento.


In [9]:
from sklearn.base import clone

resultados = {}
for peso in [None, 'balanced']:
    m = make_pipeline(clone(preprocesamiento),
                      LogisticRegression(class_weight=peso, max_iter=1000))
    m.fit(X_train, y_train)
    p = m.predict(X_valid)
    pr = m.predict_proba(X_valid)[:, 1]
    resultados[str(peso)] = {
        'accuracy': accuracy_score(y_valid, p),
        'precision': precision_score(y_valid, p, zero_division=0),
        'recall': recall_score(y_valid, p),
        'f1': f1_score(y_valid, p, zero_division=0),
        'ap': average_precision_score(y_valid, pr),
    }

pd.DataFrame(resultados).T.round(3)

,accuracy,precision,recall,f1,ap
None,0.932,0.000,0.000,0.000,0.185
balanced,0.679,0.129,0.654,0.216,0.178


In [11]:
# TODO 1. Entrene el modelo y genere predicciones.
peso_clases = 'balanced'

modelo = make_pipeline(preprocesamiento,
                       LogisticRegression(class_weight=peso_clases, max_iter=1000))
modelo.fit(X_train, y_train)
pred = modelo.predict(X_valid)

## 4. Evaluación

### Actividad 2. Métricas

Calcule Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (AP) en validación. Para las métricas AUC use las probabilidades que estima el modelo.

In [13]:
# TODO 2. Calcule las métricas y muestre la matriz de confusión.
proba = modelo.predict_proba(X_valid)[:,1]
medidas = {
    'accuracy': accuracy_score(y_valid, pred),
    'precision': precision_score(y_valid, pred, zero_division=0),
    'recall': recall_score(y_valid, pred),
    'f1': f1_score(y_valid, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, proba),
    'ap': average_precision_score(y_valid, proba),
}

for nombre, valor in medidas.items():
  print(f'{nombre}:{valor:.3f}')

print('\nMatriz de confusión (filas = real, columnas = predicho):')
print(confusion_matrix(y_valid, pred))

accuracy:0.679
precision:0.129
recall:0.654
f1:0.216
roc_auc:0.728
ap:0.178

Matriz de confusión (filas = real, columnas = predicho):
[[3334 1561]
 [ 123  232]]


### Actividad 3. Errores por segmento

Compare la tasa de error por mes de compra y estado del cliente. Muestre tanto el porcentaje de error como la cantidad de pedidos de cada segmento.

In [14]:
assert pred is not None, 'Complete primero la Actividad 1 (variable pred).'
analisis = valid.copy()
analisis['pred'] = pred
analisis['error'] = (analisis['is_late'] != analisis['pred']).astype(int)

# TODO 3. Agrupe por mes_compra y customer_state.
por_mes = (analisis.groupby('mes_compra')
           .agg(pedidos=('error', 'size'),
                tasa_error=('error', 'mean'),
                tasa_tardias=('is_late', 'mean')))
por_mes[['tasa_error', 'tasa_tardias']] = (
    por_mes[['tasa_error', 'tasa_tardias']] * 100).round(1)
print('Error por mes de compra (%):')
display(por_mes)

por_estado = (analisis.groupby('customer_state')
              .agg(pedidos=('error', 'size'),
                   tasa_error=('error', 'mean'),
                   tasa_tardias=('is_late', 'mean')))
por_estado[['tasa_error', 'tasa_tardias']] = (
    por_estado[['tasa_error', 'tasa_tardias']] * 100).round(1)
print('\nError por estado del cliente (%), ordenado por error:')
display(por_estado.sort_values('tasa_error', ascending=False))

Error por mes de compra (%):


,pedidos,tasa_error,tasa_tardias
mes_compra,,,
1,407,30.2,7.4
2,493,33.5,13.0
3,497,42.9,17.1
4,516,31.2,3.5
5,541,35.5,4.8
6,485,21.0,1.6
7,582,38.0,4.0
8,540,42.4,3.9
9,235,27.7,3.4



Error por estado del cliente (%), ordenado por error:


,pedidos,tasa_error,tasa_tardias
customer_state,,,
RR,2,100.0,0.0
PI,29,82.8,6.9
AP,4,75.0,0.0
AL,28,71.4,21.4
CE,60,70.0,15.0
BA,161,68.9,12.4
PA,51,66.7,5.9
ES,95,66.3,9.5
MA,49,65.3,26.5


### Actividad 4. Costo empresarial

Para el ejercicio, suponga un costo de R$ 8 por falso positivo y R$ 35 por falso negativo. Calcule el costo total de los errores en validación.

In [15]:
# TODO 4. Calcule el costo total con la matriz de confusión.
COSTO_FP = 8
COSTO_FN = 35

tn, fp, fn, tp = confusion_matrix(y_valid, pred).ravel()
costo_fp_fn = COSTO_FP * fp + COSTO_FN * fn

print(f'FP = {fp} | FN = {fn}')
print(f'Costo por falsos positivos: R$ {COSTO_FP * fp:,.0f}')
print(f'Costo por falsos negativos: R$ {COSTO_FN * fn:,.0f}')
print(f'Costo total del modelo:     R$ {costo_fp_fn:,.0f}')

costo_sin_modelo = COSTO_FN * (fn + tp)
print(f'Costo sin modelo (nunca alertar): R$ {costo_sin_modelo:,.0f}')

FP = 1561 | FN = 123
Costo por falsos positivos: R$ 12,488
Costo por falsos negativos: R$ 4,305
Costo total del modelo:     R$ 16,793
Costo sin modelo (nunca alertar): R$ 12,425


### Autoverificación

Ejecute estas comprobaciones después de completar las actividades.

In [16]:
assert len(muestra) == 35000, 'La muestra debe tener 35000 pedidos.'
assert (len(train), len(valid), len(test)) == (24500, 5250, 5250), (
    'Particiones esperadas: 24500 / 5250 / 5250. Revise la celda de muestra.')
assert set(train['order_id']).isdisjoint(valid['order_id']), 'Train y valid comparten pedidos.'
assert set(train['order_id']).isdisjoint(test['order_id']), 'Train y test comparten pedidos.'
assert set(valid['order_id']).isdisjoint(test['order_id']), 'Valid y test comparten pedidos.'

assert modelo is not None and pred is not None, 'Complete la Actividad 1.'
assert len(pred) == len(y_valid), 'pred debe tener 5250 predicciones de validación.'
assert set(np.unique(np.asarray(pred))).issubset({0, 1}), 'pred debe contener solo clases 0 y 1.'
assert np.array_equal(np.asarray(pred), modelo.predict(X_valid)), (
    'pred no coincide con modelo.predict(X_valid). Verifique que utilizó X_valid.')

assert all(isinstance(v, (float, int, np.number)) for v in medidas.values()), (
    'Complete las métricas de la Actividad 2.')
assert proba is not None, 'Complete la Actividad 2: calcule las probabilidades.'
assert np.asarray(proba).ndim == 1 and len(proba) == len(y_valid), (
    'proba debe contener una probabilidad por pedido de validación.')
assert np.allclose(proba, modelo.predict_proba(X_valid)[:, 1]), (
    'proba no coincide con modelo.predict_proba(X_valid)[:, 1].')
assert np.isclose(medidas['f1'], f1_score(y_valid, pred, zero_division=0)), (
    "medidas['f1'] no coincide. Use f1_score(y_valid, pred, zero_division=0).")
assert np.isclose(medidas['ap'], average_precision_score(y_valid, proba)), (
    "medidas['ap'] no coincide. Use average_precision_score(y_valid, proba).")
assert np.isclose(medidas['roc_auc'], roc_auc_score(y_valid, proba)), (
    "medidas['roc_auc'] no coincide. Use roc_auc_score(y_valid, proba).")

assert isinstance(costo_fp_fn, (int, float, np.number)), (
    'Complete la Actividad 4: costo_fp_fn debe ser numérico (8*FP + 35*FN).')
_, fp_ver, fn_ver, _ = confusion_matrix(y_valid, pred).ravel()
assert np.isclose(costo_fp_fn, 8 * fp_ver + 35 * fn_ver), (
    'El costo no coincide con 8*FP + 35*FN. Revise la Actividad 4.')
print('Autoverificación superada.')


Autoverificación superada.


## 5. Ejemplo guiado de regresión

Con los mismos predictores, estime la duración total de la entrega en días usando `Ridge` y evalúe el error absoluto medio (MAE).

In [17]:
from sklearn.base import clone

# Cada modelo recibe una copia independiente del preprocesamiento.
modelo_reg = make_pipeline(clone(preprocesamiento), Ridge(solver='lsqr'))
modelo_reg.fit(X_train, train['dias_entrega'])
pred_dias = modelo_reg.predict(X_valid)
mae = mean_absolute_error(valid['dias_entrega'], pred_dias)
print('MAE regresión:', round(mae, 2), 'días')

MAE regresión: 5.15 días


## 6. Interpretación de negocio

1. Compare la regresión logística con el modelo de referencia utilizando Accuracy, Recall y F1. Justifique su elección de `class_weight`.

Respuesta:

La elección de class weight selecionada como balanced se debe a la prueba en donde se estimo el peso la opción de None no llega a predecir sobre los retrasos de manera adecuada. ***Precision***, ***Recall*** y ***F1*** generaron resultados de ***0.000***, por lo cual el modelo no llega a detectar los pedidos tardidos. Su funcionamiento sería equiparable a un DummyClassifier.

Aunque la regresión logistica balanced tienen un ***accuracy*** menor ***0.679***, esta detecta 232 de los 355 pedidos tardíos ***Recall = 0.654***, ***F1 = 0.216***. Ademas ***Accuracy*** no llega a afectar en esta comparación significativamente debido a que los pedidos de validación tardios equivalen al 6.8% del total de los pedidos de validación de la muestra.

La regresión logistica se comporta igual a la referencia cuando se hace uso de ***class_weight=None***, nunca predice la clase tardía con el umbral de ***0.5***. El modelo penaliza mas los errores sobre la clase minoritaria y genera alertas al usar ***class_weight='balanced'***. Como consecuencia se dara una precision baja (0.129): por cada 100 pedidos señalados como tardíos, solo unos 13 lo fueron. El AP (0.178) es 2.6 veces la prevalencia (0.068) y el ROC-AUC es 0.728, lo que indica que el modelo ordena los pedidos mejor que el azar pero con una señal moderada.

2. Analice qué tipo de pedidos presenta mayor error según los segmentos estudiados. ¿Cómo se relaciona esto con el costo de falsos positivos y falsos negativos?

Respuesta:

De los 5.250 pedidos de validación, el modelo cometió 1.684 errores (32,1 %): 1.561 falsos positivos (92,7 %) y 123 falsos negativos (7,3 %). La mayor parte del error son falsas alertas, es decir, pedidos marcados como tardíos que llegaron a tiempo.

Por mes, el error es más alto en marzo (42,9 %, n = 497) y agosto (42,4 %, n = 540), seguidos de julio (38,0 %, n = 582) y mayo (35,5 %, n = 541). El más bajo es diciembre (15,0 %, n = 287). Marzo también es el mes con más retrasos reales (17,1 %), pero agosto y julio tienen error alto con solo 3,9 % y 4,0 % de retrasos reales, así que allí el error son sobre todo falsas alertas.

Por estado, entre los de al menos 100 pedidos, el error es más alto en Bahía (68,9 %, n = 161), Río de Janeiro (62,9 %, n = 687) y Santa Catarina (60,6 %, n = 180), y más bajo en São Paulo (16,1 %, n = 2.238), Paraná (15,8 %, n = 273) y Minas Gerais (20,1 %, n = 652). Las tasas extremas de estados pequeños, como Roraima (100 %, n = 2) o Amapá (75 %, n = 4), no son confiables por el tamaño de la muestra.

Esto se relaciona con el costo así: los falsos positivos cuestan R$ 8 y los falsos negativos R$ 35, pero los falsos positivos son 12,7 veces más numerosos. El costo total fue de R$ 16.793: R$ 12.488 (74,4 %) por falsos positivos y R$ 4.305 (25,6 %) por falsos negativos. Si el modelo nunca alertara, el costo sería de R$ 12.425 (355 pedidos tardíos × R$ 35), es decir, R$ 4.368 menos. Alertar compensa solo si la probabilidad de retraso supera 8 / (8 + 35) ≈ 18,6 %, y la Precision del modelo (0,129) está por debajo. Estos costos son supuestos del ejercicio, no costos reales de Olist.

3. Proponga una mejora concreta para la semana 13. Considere la información del producto, del vendedor, del pago o el tratamiento de variables categóricas. Explique cómo verificará la mejora usando la misma partición de validación.

Respuesta:

Propongo ajustar el umbral de decisión usando el costo de negocio, en lugar de dejar el 0.5 por defecto. El modelo actual ya produce probabilidades (predict_proba) con un ROC-AUC de 0.728, así que sirven para ordenar los pedidos por riesgo. La hipótesis es que existe un umbral más alto en el que solo se alerten los pedidos más riesgosos, de modo que la Precision supere el punto de equilibrio de 18.6 % y el costo total baje del valor actual de R$ 16.793, e idealmente por debajo de los R$ 12.425 que cuesta no alertar nunca.

Para comprobarlo usaré la misma partición de validación (mismo ID, misma semilla y mismo X_valid). Probaré umbrales entre 0.10 y 0.90, y para cada uno calcularé FP, FN, Precision, Recall, F1 y el costo 8·FP + 35·FN, sin volver a entrenar el modelo. Compararé el mejor umbral contra la línea base de la semana 12 (costo R$ 16.793, F1 0.216). Considero que hay mejora si el costo baja de forma clara y el F1 o el Recall no se desploman. Como el umbral se elige sobre validación, el resultado puede ser algo optimista, por lo que no tocaré la partición de prueba hasta la semana 16.

Como segunda línea para comparar, probaría un modelo no lineal (por ejemplo, gradient boosting), porque el AP bajo sugiere que las relaciones entre las variables y el retraso pueden no ser lineales, y evaluaría si mejora el AP y el ROC-AUC en la misma validación.

## 7. Registro de resultados

Guarde el resultado del laboratorio y la huella de la muestra. Las indicaciones de entrega están en el documento técnico.

In [18]:
import sys
import sklearn

reporte = {
    'laboratorio': 'S12',
    'id_estudiante': identificador,
    'semilla': semilla,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__,
    'n_muestra': len(muestra),
    'n_train': len(train),
    'n_valid': len(valid),
    'n_test': len(test),
    'n_variables': len(columnas),
    'columnas_predictoras': columnas,
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'tasa_tardias_muestra': float(muestra['is_late'].mean()),
    'huella_muestra': huella_muestra,
    'peso_clases': str(peso_clases),
    'metricas': {nombre: float(valor) for nombre, valor in medidas.items()},
    'costo_errores_brl': float(costo_fp_fn),
    'mae_regresion': float(mae),
}

ruta = Path(f'reports/S12_resultados_ID_{identificador}.json')
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reporte:', ruta)

# En Colab, descarga el JSON para conservarlo fuera de la sesión.
try:
    from google.colab import files
    files.download(str(ruta))
except Exception:
    print('Si no se descargó automáticamente, guarde manualmente:', ruta)


Reporte: reports/S12_resultados_ID_05.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>